# 05. `stress_level` 예측 대치 (Predictive Imputation)

지금까지 확인된 병목: `stress_level`은 라벨 생성 규칙의 핵심 3피처 중 유일하게 **뚜렷한 대리변수가 없어서 복구를 못 하고 있는** 피처였음(가이드 문서 "이게 이 대회의 실질적인 벽"). 단, 이건 피처 하나씩 봤을 때 얘기이고, 여러 피처를 동시에 조합하면 약한 신호가 잡힐 수도 있습니다.

**방법**: `stress_level`이 관측된 행으로 보조 분류기(LightGBM)를 학습해서 `low/medium/high` 확률을 예측하고, 이 확률 3개를 새 피처(`stress_proba_low/medium/high`)로 메인 모델에 추가합니다.

**리키지 방지**: 보조 분류기도 메인 모델과 동일한 5-fold(`cv_folds.csv`)를 그대로 재사용해서, 각 fold의 검증셋에 들어가는 행에 대한 확률은 반드시 **그 fold를 제외한 나머지로 학습한 모델**에서 뽑습니다 (OOF 방식). test는 train 전체로 학습한 모델로 예측.

커널: **Python (teammate)**

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.metrics import balanced_accuracy_score, accuracy_score, log_loss
import lightgbm as lgb

SEED = 42
N_FOLDS = 5
DATA_DIR = Path("../playground-series-s6e7")
OUT_DIR = DATA_DIR / "processed"
TARGET = "health_condition"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
folds = pd.read_csv(OUT_DIR / "cv_folds.csv")
train = train.merge(folds, on="id", how="left")
assert train["fold"].isna().sum() == 0

print(train["stress_level"].value_counts(dropna=False, normalize=True).round(4))

stress_level
medium    0.3794
high      0.2576
low       0.2430
NaN       0.1200
Name: proportion, dtype: float64


## 1. 04와 동일한 결측 복구 피처 (baseline 재사용)

In [2]:
def add_recovery_features(df, step_tertiles, sleep_quality_cond):
    df = df.copy()
    activity_proxy = pd.cut(
        df["step_count"], bins=[-np.inf, step_tertiles[0], step_tertiles[1], np.inf],
        labels=["sedentary", "moderate", "active"],
    ).astype(object)
    df["physical_activity_level_recovered"] = df["physical_activity_level"]
    missing_activity = df["physical_activity_level"].isna()
    df.loc[missing_activity, "physical_activity_level_recovered"] = activity_proxy[missing_activity]
    df["physical_activity_level_recovered"] = df["physical_activity_level_recovered"].fillna("moderate")

    df["sleep_duration_recovered"] = df["sleep_duration"]
    missing_sleep = df["sleep_duration"].isna()
    fallback_median = sleep_quality_cond.get("missing", sleep_quality_cond["average"])
    mapped = df.loc[missing_sleep, "sleep_quality"].map(sleep_quality_cond).fillna(fallback_median)
    df.loc[missing_sleep, "sleep_duration_recovered"] = mapped

    df["stress_level_isnull"] = df["stress_level"].isna().astype(np.int8)
    df["sleep_duration_isnull"] = df["sleep_duration"].isna().astype(np.int8)
    df["physical_activity_level_isnull"] = df["physical_activity_level"].isna().astype(np.int8)
    return df


step_tertiles = train["step_count"].quantile([1/3, 2/3]).values
sleep_quality_cond = train.groupby("sleep_quality")["sleep_duration"].median().to_dict()
sleep_quality_cond["missing"] = train["sleep_duration"].median()

train = add_recovery_features(train, step_tertiles, sleep_quality_cond)
test = add_recovery_features(test, step_tertiles, sleep_quality_cond)

## 2. 보조 분류기용 피처셋 구성

`stress_level`을 예측하는 데 쓸 입력 피처: `stress_level` 자신과 `health_condition`(메인 타겟)을 제외한 나머지 전부. `sleep_duration_recovered`, `physical_activity_level_recovered`까지 포함(이미 결측이 채워진 상태라 결측 없이 쓸 수 있음).

In [3]:
AUX_NUMERIC = [
    "sleep_duration_recovered", "heart_rate", "bmi", "calorie_expenditure",
    "step_count", "exercise_duration", "water_intake",
]
AUX_ORDINAL = {
    "sleep_quality": ["poor", "average", "good"],
    "physical_activity_level_recovered": ["sedentary", "moderate", "active"],
    "smoking_alcohol": ["no", "occasional", "yes"],
}
AUX_NOMINAL = ["diet_type", "gender"]
AUX_FLAGS = ["sleep_duration_isnull", "physical_activity_level_isnull"]

# 원본 train/test를 건드리지 않도록 별도 복사본에서 인코딩 (메인 모델 섹션에서 같은 컬럼을
# 다시 인코딩해야 하므로, 여기서 train/test 자체를 수정하면 충돌남)
aux_train_raw = train[AUX_NUMERIC + list(AUX_ORDINAL.keys()) + AUX_NOMINAL + AUX_FLAGS].copy()
aux_test_raw = test[AUX_NUMERIC + list(AUX_ORDINAL.keys()) + AUX_NOMINAL + AUX_FLAGS].copy()

aux_numeric_medians = aux_train_raw[AUX_NUMERIC].median()
for df in (aux_train_raw, aux_test_raw):
    for col in AUX_NUMERIC:
        df[col] = df[col].fillna(aux_numeric_medians[col])

aux_categorical_cols = list(AUX_ORDINAL.keys()) + AUX_NOMINAL
for df in (aux_train_raw, aux_test_raw):
    for col in aux_categorical_cols:
        df[col] = df[col].fillna("missing")

for col, order in AUX_ORDINAL.items():
    categories = order + ["missing"]
    enc = OrdinalEncoder(categories=[categories])
    aux_train_raw[col] = enc.fit_transform(aux_train_raw[[col]])
    aux_test_raw[col] = enc.transform(aux_test_raw[[col]])

aux_train_ohe = pd.get_dummies(aux_train_raw[AUX_NOMINAL], prefix=[f"aux_{c}" for c in AUX_NOMINAL])
aux_test_ohe = pd.get_dummies(aux_test_raw[AUX_NOMINAL], prefix=[f"aux_{c}" for c in AUX_NOMINAL])
aux_test_ohe = aux_test_ohe.reindex(columns=aux_train_ohe.columns, fill_value=0)

AUX_FEATURE_COLS = AUX_NUMERIC + list(AUX_ORDINAL.keys()) + AUX_FLAGS + list(aux_train_ohe.columns)

aux_train_X = pd.concat([aux_train_raw[AUX_NUMERIC + list(AUX_ORDINAL.keys()) + AUX_FLAGS], aux_train_ohe], axis=1)
aux_test_X = pd.concat([aux_test_raw[AUX_NUMERIC + list(AUX_ORDINAL.keys()) + AUX_FLAGS], aux_test_ohe], axis=1)

stress_encoder = LabelEncoder()
stress_encoder.fit(["low", "medium", "high"])
print("stress class order:", list(stress_encoder.classes_))
print(len(AUX_FEATURE_COLS), "aux features")

stress class order: [np.str_('high'), np.str_('low'), np.str_('medium')]
20 aux features


## 3. OOF로 보조 분류기 학습 + 확률 피처 생성 (누수 방지)

In [4]:
stress_observed = train["stress_level"].notna()
oof_stress_proba = np.full((len(train), 3), np.nan)

aux_params = dict(
    objective="multiclass", num_class=3, n_estimators=500,
    learning_rate=0.05, num_leaves=63, subsample=0.8,
    colsample_bytree=0.8, random_state=SEED, verbosity=-1,
)

fold_val_scores = []
for fold in range(N_FOLDS):
    aux_tr_mask = (train["fold"] != fold) & stress_observed
    X_tr = aux_train_X.loc[aux_tr_mask, AUX_FEATURE_COLS]
    y_tr = stress_encoder.transform(train.loc[aux_tr_mask, "stress_level"])

    # 검증용: 같은 fold 중 stress 관측된 행 (모델 품질 체크)
    val_mask = (train["fold"] == fold) & stress_observed
    X_val = aux_train_X.loc[val_mask, AUX_FEATURE_COLS]
    y_val = stress_encoder.transform(train.loc[val_mask, "stress_level"])

    model = lgb.LGBMClassifier(**aux_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])

    val_pred = model.predict(X_val)
    val_acc = accuracy_score(y_val, val_pred)
    val_ll = log_loss(y_val, model.predict_proba(X_val), labels=[0, 1, 2])
    fold_val_scores.append((val_acc, val_ll))
    print(f"fold {fold}: stress_level 예측 accuracy={val_acc:.4f}, log_loss={val_ll:.4f}")

    # 이 fold 전체(관측+결측 모두)에 대해 확률 예측 -> OOF 배열에 채움
    fold_all_mask = train["fold"] == fold
    oof_stress_proba[fold_all_mask.values] = model.predict_proba(aux_train_X.loc[fold_all_mask, AUX_FEATURE_COLS])

avg_acc = np.mean([s[0] for s in fold_val_scores])
baseline_majority_acc = train.loc[stress_observed, "stress_level"].value_counts(normalize=True).max()
print(f"\n평균 accuracy: {avg_acc:.4f}  (기준선: 최빈 클래스만 찍었을 때 {baseline_majority_acc:.4f})")

fold 0: stress_level 예측 accuracy=0.4619, log_loss=1.0453


fold 1: stress_level 예측 accuracy=0.4619, log_loss=1.0451


fold 2: stress_level 예측 accuracy=0.4607, log_loss=1.0452


fold 3: stress_level 예측 accuracy=0.4586, log_loss=1.0462


fold 4: stress_level 예측 accuracy=0.4616, log_loss=1.0448



평균 accuracy: 0.4609  (기준선: 최빈 클래스만 찍었을 때 0.4311)


## 4. 관측된 행은 실제값(one-hot)으로, 결측 행만 모델 예측값 사용

In [5]:
PROBA_COLS = ["stress_proba_low", "stress_proba_medium", "stress_proba_high"]

train[PROBA_COLS] = oof_stress_proba[:, [list(stress_encoder.classes_).index(c) for c in ["low", "medium", "high"]]]

# 관측된 행은 실제값 one-hot으로 덮어쓰기 (확실한 정보이므로 모델 추정치보다 우선)
true_stress_map = {"low": [1, 0, 0], "medium": [0, 1, 0], "high": [0, 0, 1]}
observed_idx = train.index[stress_observed]
train.loc[observed_idx, PROBA_COLS] = np.array(
    [true_stress_map[v] for v in train.loc[observed_idx, "stress_level"]]
)

# test용: train 전체(stress 관측)로 재학습한 모델로 예측, 관측 행은 실제값 사용
final_aux_model = lgb.LGBMClassifier(**aux_params)
final_aux_model.fit(
    aux_train_X.loc[stress_observed, AUX_FEATURE_COLS],
    stress_encoder.transform(train.loc[stress_observed, "stress_level"]),
)
test_stress_proba = final_aux_model.predict_proba(aux_test_X[AUX_FEATURE_COLS])
test[PROBA_COLS] = test_stress_proba[:, [list(stress_encoder.classes_).index(c) for c in ["low", "medium", "high"]]]

test_stress_observed = test["stress_level"].notna()
test_observed_idx = test.index[test_stress_observed]
test.loc[test_observed_idx, PROBA_COLS] = np.array(
    [true_stress_map[v] for v in test.loc[test_observed_idx, "stress_level"]]
)

train[["stress_level"] + PROBA_COLS].head(10)

,stress_level,stress_proba_low,stress_proba_medium,stress_proba_high
0,high,0.00000,0.00000,1.00000
1,low,1.00000,0.00000,0.00000
2,high,0.00000,0.00000,1.00000
3,high,0.00000,0.00000,1.00000
4,NaN,0.23208,0.44305,0.32487
5,low,1.00000,0.00000,0.00000
6,low,1.00000,0.00000,0.00000
7,low,1.00000,0.00000,0.00000
8,high,0.00000,0.00000,1.00000
9,medium,0.00000,1.00000,0.00000


## 5. 메인 모델(04와 동일 구조)에 확률 피처 3개 추가해서 재검증

In [6]:
NUMERIC_COLS = [
    "sleep_duration_recovered", "heart_rate", "bmi", "calorie_expenditure",
    "step_count", "exercise_duration", "water_intake",
]
ORDINAL_COLS = {
    "stress_level": ["low", "medium", "high"],
    "sleep_quality": ["poor", "average", "good"],
    "physical_activity_level_recovered": ["sedentary", "moderate", "active"],
    "smoking_alcohol": ["no", "occasional", "yes"],
}
NOMINAL_COLS = ["diet_type", "gender"]
FLAG_COLS = ["stress_level_isnull", "sleep_duration_isnull", "physical_activity_level_isnull"]

numeric_medians = train[NUMERIC_COLS].median()
for df in (train, test):
    for col in NUMERIC_COLS:
        df[col] = df[col].fillna(numeric_medians[col])

categorical_cols = list(ORDINAL_COLS.keys()) + NOMINAL_COLS
for df in (train, test):
    for col in categorical_cols:
        df[col] = df[col].fillna("missing")

for col, order in ORDINAL_COLS.items():
    categories = order + ["missing"]
    encoder = OrdinalEncoder(categories=[categories])
    train[col] = encoder.fit_transform(train[[col]])
    test[col] = encoder.transform(test[[col]])

train_ohe = pd.get_dummies(train[NOMINAL_COLS], prefix=NOMINAL_COLS)
test_ohe = pd.get_dummies(test[NOMINAL_COLS], prefix=NOMINAL_COLS)
test_ohe = test_ohe.reindex(columns=train_ohe.columns, fill_value=0)
train = pd.concat([train.drop(columns=NOMINAL_COLS), train_ohe], axis=1)
test = pd.concat([test.drop(columns=NOMINAL_COLS), test_ohe], axis=1)

FEATURE_COLS_V2 = (
    NUMERIC_COLS + list(ORDINAL_COLS.keys()) + FLAG_COLS + list(train_ohe.columns) + PROBA_COLS
)

target_encoder = LabelEncoder()
train["target_enc"] = target_encoder.fit_transform(train[TARGET])
lgb_class_order = list(target_encoder.classes_)
train_priors = train[TARGET].value_counts(normalize=True).to_dict()

print(len(FEATURE_COLS_V2), "features (04 대비 +3:", PROBA_COLS, ")")

25 features (04 대비 +3: ['stress_proba_low', 'stress_proba_medium', 'stress_proba_high'] )


In [7]:
def prior_corrected_predict(proba, class_order, priors):
    prior_arr = np.array([priors[c] for c in class_order])
    scores = proba / prior_arr
    return np.array(class_order)[scores.argmax(axis=1)]


main_params = dict(
    objective="multiclass", num_class=3, n_estimators=500,
    learning_rate=0.05, num_leaves=63, subsample=0.8,
    colsample_bytree=0.8, random_state=SEED, verbosity=-1,
)

oof_proba_v2 = np.zeros((len(train), 3))
for fold in range(N_FOLDS):
    tr_idx = train["fold"] != fold
    va_idx = train["fold"] == fold
    X_tr, y_tr = train.loc[tr_idx, FEATURE_COLS_V2], train.loc[tr_idx, "target_enc"]
    X_va, y_va = train.loc[va_idx, FEATURE_COLS_V2], train.loc[va_idx, "target_enc"]

    model = lgb.LGBMClassifier(**main_params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_proba_v2[va_idx.values] = model.predict_proba(X_va)
    print(f"fold {fold} done")

pred_v2 = prior_corrected_predict(oof_proba_v2, lgb_class_order, train_priors)
ba_v2 = balanced_accuracy_score(train[TARGET].values, pred_v2)
acc_v2 = accuracy_score(train[TARGET].values, pred_v2)

print(f"\n04 baseline(사전확률 보정)    : 0.94980")
print(f"04 tuned                      : 0.94987")
print(f"05 stress_proba 추가(사전보정) : {ba_v2:.5f}  (accuracy {acc_v2:.5f})")
print(f"개선폭 (04 baseline 대비)      : {ba_v2 - 0.94980:+.5f}")

fold 0 done


fold 1 done


fold 2 done


fold 3 done


fold 4 done



04 baseline(사전확률 보정)    : 0.94980
04 tuned                      : 0.94987
05 stress_proba 추가(사전보정) : 0.94955  (accuracy 0.93939)
개선폭 (04 baseline 대비)      : -0.00025


## 6. Feature Importance (gain) — stress_proba 피처가 실제로 쓰이는지 확인

In [8]:
gain_model = lgb.LGBMClassifier(**main_params, importance_type="gain")
gain_model.fit(train[FEATURE_COLS_V2], train["target_enc"])
importance = pd.Series(gain_model.feature_importances_, index=FEATURE_COLS_V2).sort_values(ascending=False)
importance.head(15)

sleep_duration_recovered             1.847575e+06
stress_proba_high                    1.217501e+06
physical_activity_level_recovered    8.841469e+05
stress_proba_low                     7.823320e+05
stress_proba_medium                  4.880554e+05
stress_level                         3.747962e+05
sleep_duration_isnull                1.733401e+05
bmi                                  1.285393e+05
exercise_duration                    7.703620e+04
step_count                           7.442748e+04
water_intake                         5.103895e+04
calorie_expenditure                  4.760044e+04
heart_rate                           4.711790e+04
physical_activity_level_isnull       3.145384e+04
sleep_quality                        2.657527e+04
dtype: float64

## 7. 결측 패턴별 개선 여부 확인 (stress 결측 행에서만 따로 채점)

In [9]:
stress_missing_mask = train["stress_level_isnull"] == 1

ba_stress_missing_v2 = balanced_accuracy_score(
    train.loc[stress_missing_mask, TARGET], pred_v2[stress_missing_mask.values]
)
print(f"stress_level 결측 행({stress_missing_mask.sum()}개)만 따로 본 BA: {ba_stress_missing_v2:.5f}")
print("(가이드 문서 기준 이전 값: stress만 결측 0.8771, 04 이전 전체 결측 패턴 분해 참고)")

stress_level 결측 행(82811개)만 따로 본 BA: 0.85966
(가이드 문서 기준 이전 값: stress만 결측 0.8771, 04 이전 전체 결측 패턴 분해 참고)


## 8. 최종 제출 파일 생성 (개선됐을 경우에만 채택)

In [10]:
if ba_v2 > 0.94987:
    final_model = lgb.LGBMClassifier(**main_params)
    final_model.fit(train[FEATURE_COLS_V2], train["target_enc"])
    test_proba = final_model.predict_proba(test[FEATURE_COLS_V2])
    test_pred = prior_corrected_predict(test_proba, lgb_class_order, train_priors)
    submission = pd.DataFrame({"id": test["id"], TARGET: test_pred})
    submission.to_csv(OUT_DIR / "submission_v3_stress_proba.csv", index=False)
    print("개선 확인, 저장:", OUT_DIR / "submission_v3_stress_proba.csv")
    print(submission[TARGET].value_counts(normalize=True))
else:
    print("04 대비 개선 없음 -> submission_v2_tuned.csv를 그대로 유지")

04 대비 개선 없음 -> submission_v2_tuned.csv를 그대로 유지
